# Image-based modeling workflow

Step through each cell. You decide when to move on.

1. Load `image.tif` and preview it.
2. Detect the scale bar in the bottom-left, confirm its pixel length, enter the physical length (e.g. `100 nm`).
3. Build the modeling region (everything opaque, minus the scale-bar area). This is treated as the *full* calculation domain.
4. Segment by colour and look at the candidates.
5. Pick each region from a dropdown and give it a name.
6. Save an annotated PNG and a per-region summary.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

import image_modeling as im

## 1. Load the image

In [ ]:
IMAGE_PATH = 'image.tif'
image = im.load_image(IMAGE_PATH)
print('shape:', image.shape, 'dtype:', image.dtype)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(image)
ax.set_title('input image')
ax.axis('off')
plt.show()

## 2. Detect the scale bar and set its physical length

The detector searches the bottom-left quadrant for opaque black pixels and
picks the row with the longest horizontal run as the bar. Confirm the red
highlight sits on the actual bar, then type the physical length (e.g. `100`)
and the unit (default `nm`).

In [ ]:
scale_bar = im.detect_scale_bar(image)
print('scale bar bbox (y0,y1,x0,x1):', scale_bar.bbox)
print('bar row:', scale_bar.bar_row,
      'x:', scale_bar.bar_x0, '..', scale_bar.bar_x1,
      'pixel length:', scale_bar.pixel_length)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(image)
y0, y1, x0, x1 = scale_bar.bbox
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec='red', lw=1.5))
ax.plot([scale_bar.bar_x0, scale_bar.bar_x1],
        [scale_bar.bar_row, scale_bar.bar_row], 'r-', lw=3)
ax.set_title('detected scale bar')
ax.axis('off')
plt.show()

In [ ]:
PHYSICAL_LENGTH = 100   # <-- edit me
UNIT = 'nm'             # <-- edit me

im.set_physical_length(scale_bar, PHYSICAL_LENGTH, UNIT)
print(f'scale = {scale_bar.scale:.4f} {UNIT}/pixel')

## 3. Build the modeling region

Everything opaque minus the scale-bar bounding box. The green overlay shows
what will be treated as the full calculation domain.

In [ ]:
modeling_mask = im.build_modeling_mask(image, scale_bar)
print('modeling pixels:', int(modeling_mask.sum()))

overlay = np.zeros((*modeling_mask.shape, 4), dtype=np.uint8)
overlay[modeling_mask] = (0, 255, 0, 90)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(image)
ax.imshow(overlay)
ax.set_title('modeling region (green)')
ax.axis('off')
plt.show()

## 4. Segment by colour

Each distinct RGB colour inside the modeling region becomes a candidate
region. They are indexed in order of pixel count.

In [ ]:
regions = im.segment_by_color(image, modeling_mask)
print(f'found {len(regions)} colour regions')

cols = min(len(regions), 4)
rows = int(np.ceil(len(regions) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows), squeeze=False)
for ax, r in zip(axes.flat, regions):
    view = image.copy()
    view[~r.mask] = (0, 0, 0, 0)
    ax.imshow(view)
    cy, cx = r.centroid()
    ax.set_title(f'#{r.index}  rgb={r.color}\npx={r.pixel_count}')
    ax.axis('off')
for ax in axes.flat[len(regions):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Name the regions

Pick a region index, type a name, click **Assign**. Repeat as many times as
you like; you can also re-assign. Un-named regions will be left out of the
final annotation. Hit **Show current names** any time to check state.

In [ ]:
region_selector = widgets.Dropdown(
    options=[(f'#{r.index}  rgb={r.color}  px={r.pixel_count}', r.index) for r in regions],
    description='region:',
)
name_input = widgets.Text(description='name:', placeholder='e.g. Si, Al, Au, P-Si')
assign_btn = widgets.Button(description='Assign', button_style='primary')
show_btn = widgets.Button(description='Show current names')
clear_btn = widgets.Button(description='Clear')
out = widgets.Output()

def _region_by_index(idx):
    for r in regions:
        if r.index == idx:
            return r
    return None

def _on_assign(_):
    r = _region_by_index(region_selector.value)
    if r is None:
        return
    r.name = name_input.value.strip() or None
    with out:
        out.clear_output()
        print(f'region #{r.index} -> {r.name!r}')

def _on_show(_):
    with out:
        out.clear_output()
        for r in regions:
            print(f'  #{r.index} rgb={r.color} px={r.pixel_count} name={r.name!r}')

def _on_clear(_):
    r = _region_by_index(region_selector.value)
    if r is not None:
        r.name = None
    with out:
        out.clear_output()
        print(f'cleared name for region #{region_selector.value}')

assign_btn.on_click(_on_assign)
show_btn.on_click(_on_show)
clear_btn.on_click(_on_clear)

display(widgets.VBox([
    widgets.HBox([region_selector, name_input]),
    widgets.HBox([assign_btn, clear_btn, show_btn]),
    out,
]))

## 6. Annotate and save

The output PNG shows every *named* region with its label, plus the scale
bar with its physical length written above it.

In [ ]:
ctx = im.ModelingContext(
    image=image,
    scale_bar=scale_bar,
    modeling_mask=modeling_mask,
    regions=regions,
)
out_path = im.annotate(ctx, 'image_annotated.png')
print('wrote', out_path)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(plt.imread(out_path))
ax.set_title('annotated output')
ax.axis('off')
plt.show()

In [ ]:
summary = im.summarise(ctx)
for name, info in summary.items():
    print(name)
    for k, v in info.items():
        print(f'  {k}: {v}')